# Plasmid Host-Genus Prediction — Full Training on Colab (v2)

**Improvements over v1:**
- **6 random windows per plasmid per epoch** (was 1) — model sees 6× more sequence
- **3000 bp windows** (was 1000) — 3× more context per window
- **Rare-class oversampling** — WeightedRandomSampler balances genera during training
- **Cosine LR schedule** — better fine-tuning convergence
- **10 epochs** (was 5) — more training with the larger effective dataset

**What this notebook does:**
1. Clones the repo and installs dependencies
2. Downloads PLSDB data from Figshare (~2 GB) and preprocesses it
3. Mounts Google Drive for checkpoint persistence
4. Fine-tunes Nucleotide Transformer v2 (100M) on 62k+ plasmids
5. Auto-saves checkpoints to Drive after every epoch (survives Colab disconnects)
6. Evaluates the best checkpoint on the test set
7. Runs a sample prediction

**Runtime requirements:** GPU runtime (T4 is fine, A100 is faster).

**Estimated time:** 4–6 hours on A100, 8–12 hours on T4.

---

## 0. Safety: Verify GPU and mount Google Drive

In [ ]:
# ====== GPU CHECK ======
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime > Change runtime type > GPU (T4 or A100).\n"
        "Training on CPU would take days — do not proceed without a GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ====== MOUNT GOOGLE DRIVE ======
# All checkpoints and results are saved here so they survive disconnects.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/plasmid-host-range'
DRIVE_CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT_DIR, 'checkpoints')
DRIVE_REPORTS_DIR = os.path.join(DRIVE_PROJECT_DIR, 'reports')
DRIVE_DATA_DIR = os.path.join(DRIVE_PROJECT_DIR, 'data_processed')

for d in [DRIVE_CHECKPOINT_DIR, DRIVE_REPORTS_DIR, DRIVE_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Drive project dir: {DRIVE_PROJECT_DIR}")
print("Checkpoints will be saved to Drive after every epoch.")

## 1. Clone repo and install dependencies

In [ ]:
# ====== CLONE AND INSTALL ======
%cd /content
!rm -rf plasmid-host-range
!git clone https://github.com/QuercusCode/Plasmid-host-range.git plasmid-host-range
%cd /content/plasmid-host-range

# Install the package + deps (transformers pinned to 4.x by pyproject.toml)
!pip install -e '.[dev]' -q

# Ensure the package is importable even if editable install doesn't register
# immediately in Colab's kernel (a known Colab quirk).
import sys
if '/content/plasmid-host-range/src' not in sys.path:
    sys.path.insert(0, '/content/plasmid-host-range/src')

# Verify
import transformers, accelerate, tokenizers
print(f"transformers: {transformers.__version__}")
print(f"accelerate: {accelerate.__version__}")
print(f"tokenizers: {tokenizers.__version__}")

# Quick import check
from plasmid_host_range import __version__
print(f"plasmid_host_range: {__version__} ✓")

## 2. Download PLSDB data from Figshare

Downloads ~2 GB (mostly the FASTA). Then decompresses to ~7 GB.
If this cell was already run and data exists, it skips automatically.

In [ ]:
# ====== CHECK FOR CACHED PROCESSED DATA ON DRIVE ======
# If you already ran preprocessing in a previous session and it's on Drive,
# skip the expensive download + preprocess step.
import shutil

PROCESSED_LOCAL = '/content/plasmid-host-range/data/processed'
cached_parquets = [
    os.path.join(DRIVE_DATA_DIR, f) for f in ['train.parquet', 'val.parquet', 'test.parquet', 'label_names.json']
]

if all(os.path.exists(f) for f in cached_parquets):
    print("Found cached processed data on Drive — copying to local disk (faster I/O)...")
    os.makedirs(PROCESSED_LOCAL, exist_ok=True)
    for f in cached_parquets:
        shutil.copy2(f, PROCESSED_LOCAL)
    print("Done. Skipping download + preprocess.")
    SKIP_DOWNLOAD = True
else:
    print("No cached data on Drive — will download and preprocess from scratch.")
    SKIP_DOWNLOAD = False

In [ ]:
# ====== DOWNLOAD (if needed) ======
if not SKIP_DOWNLOAD:
    !plasmid-host-range download
    print("\nDownload complete.")
    !ls -lh data/raw/

In [ ]:
# ====== PREPROCESS (if needed) ======
if not SKIP_DOWNLOAD:
    !plasmid-host-range preprocess --top-n-genera 20

    # Cache processed splits to Drive for future sessions
    print("\nCaching processed data to Drive for future sessions...")
    for fname in ['train.parquet', 'val.parquet', 'test.parquet', 'label_names.json']:
        src = os.path.join(PROCESSED_LOCAL, fname)
        dst = os.path.join(DRIVE_DATA_DIR, fname)
        if os.path.exists(src):
            shutil.copy2(src, dst)
    print("Cached to Drive.")

# Verify
!ls -lh data/processed/

## 3. Disk and memory check before training

In [ ]:
# ====== RESOURCE CHECK ======
import psutil

disk = shutil.disk_usage('/content')
ram = psutil.virtual_memory()

print(f"Disk free:  {disk.free / 1e9:.1f} GB")
print(f"RAM:        {ram.total / 1e9:.1f} GB total, {ram.available / 1e9:.1f} GB available")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

if disk.free < 10e9:
    print("\n\u26a0\ufe0f WARNING: Less than 10 GB disk free. Consider deleting data/raw/ after preprocessing.")
    print("   Run: !rm -rf data/raw/")
if ram.available < 8e9:
    print("\n\u26a0\ufe0f WARNING: Less than 8 GB RAM available. Training may OOM on data loading.")

## 4. Configure training

This cell writes a Colab-optimised YAML config. Key differences from the default:
- Checkpoints are saved to **Google Drive** (survive disconnects)
- `fp16: true` for speed on T4/A100
- Slightly larger batch size (GPU has more VRAM than a laptop)
- Auto-resume from the latest checkpoint if the session was interrupted

In [ ]:
# ====== WRITE COLAB V2 CONFIG ======
COLAB_CONFIG = '/content/plasmid-host-range/configs/colab_v2.yaml'

config_content = f"""
# Improved Colab config (v2). See configs/colab_v2.yaml for documentation.
model_name: InstaDeepAI/nucleotide-transformer-v2-100m-multi-species
trust_remote_code: true

data:
  processed_dir: data/processed
  window_size: 3000
  train_windows_per_plasmid: 6
  eval_windows_per_plasmid: 10
  max_tokens: 512

training:
  output_dir: {DRIVE_CHECKPOINT_DIR}/colab_v2_run
  num_train_epochs: 10
  per_device_train_batch_size: 16
  per_device_eval_batch_size: 32
  learning_rate: 2.0e-5
  weight_decay: 0.01
  warmup_ratio: 0.06
  lr_scheduler_type: cosine
  eval_strategy: epoch
  save_strategy: epoch
  save_total_limit: 3
  load_best_model_at_end: true
  metric_for_best_model: macro_f1
  greater_is_better: true
  class_weighted_loss: true
  oversample_rare_classes: true
  seed: 42
  fp16: true

labels:
  top_n_genera: 20
  other_label: Other
""".strip()

with open(COLAB_CONFIG, 'w') as f:
    f.write(config_content)

print("Colab v2 config written.")
print(f"Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}/colab_v2_run")
print()
print("Key improvements over v1:")
print("  - 6 windows/plasmid/epoch (was 1) = 6x more sequence coverage")
print("  - 3000 bp windows (was 1000) = 3x more context")
print("  - Rare-class oversampling = balanced genus exposure")
print("  - Cosine LR schedule")
print("  - 10 epochs (was 5)")

## 5. Train (with auto-resume)

**Safety features:**
- Checkpoints saved to Google Drive after every epoch
- If the Colab session disconnects and you reconnect, re-run from Cell 0 —
  it will detect cached data on Drive, skip download/preprocess, and auto-resume
  training from the last saved checkpoint.
- `save_total_limit: 3` keeps only the 3 most recent checkpoints to avoid
  filling your Drive.

In [ ]:
# ====== AUTO-RESUME LOGIC ======
# Check if there's an existing checkpoint from a previous interrupted run.
import glob

checkpoint_dir = os.path.join(DRIVE_CHECKPOINT_DIR, 'colab_v2_run')
existing_ckpts = sorted(glob.glob(os.path.join(checkpoint_dir, 'checkpoint-*')))

if existing_ckpts:
    latest = existing_ckpts[-1]
    print(f"\n\u2705 Found existing checkpoint: {latest}")
    print("Training will RESUME from this checkpoint.")
    print(f"(Found {len(existing_ckpts)} checkpoint(s) total)\n")
    RESUME_FROM = latest
else:
    print("No existing checkpoints found. Training from scratch.")
    RESUME_FROM = None

In [ ]:
# ====== TRAINING ======
import json
import yaml
import numpy as np
import functools
from pathlib import Path

# ---- PyTorch 2.6 compatibility fix ----
# PyTorch 2.6 changed torch.load to default weights_only=True.
# HuggingFace Trainer saves numpy arrays in optimizer/scheduler state,
# which trips this restriction when resuming from a checkpoint.
# We restore the old default — safe because WE created these checkpoints.
torch.load = functools.partial(torch.load, weights_only=False)

from plasmid_host_range.data.dataset import PlasmidWindowDataset, build_oversampler
from plasmid_host_range.data.splits import compute_class_weights
from plasmid_host_range.model import load_model

from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments
import torch.nn as nn

# Load config
with open(COLAB_CONFIG) as f:
    cfg = yaml.safe_load(f)

data_cfg = cfg['data']
train_cfg = cfg['training']
processed = Path(data_cfg['processed_dir'])

label_names = json.loads((processed / 'label_names.json').read_text())
num_labels = len(label_names)
print(f"Classes: {num_labels} — {label_names}")

# Load model
loaded = load_model(
    cfg['model_name'],
    num_labels=num_labels,
    trust_remote_code=cfg.get('trust_remote_code', True),
)
print(f"Model: {cfg['model_name']}")
print(f"Parameters: {sum(p.numel() for p in loaded.model.parameters()):,}")

# Build datasets
train_ds = PlasmidWindowDataset(
    processed / 'train.parquet',
    tokenizer=loaded.tokenizer,
    window_size=data_cfg['window_size'],
    max_tokens=data_cfg['max_tokens'],
    mode='train',
    train_windows_per_plasmid=data_cfg.get('train_windows_per_plasmid', 1),
    seed=train_cfg.get('seed', 42),
)
val_ds = PlasmidWindowDataset(
    processed / 'val.parquet',
    tokenizer=loaded.tokenizer,
    window_size=data_cfg['window_size'],
    max_tokens=data_cfg['max_tokens'],
    mode='eval',
    eval_windows_per_plasmid=data_cfg['eval_windows_per_plasmid'],
    seed=train_cfg.get('seed', 42),
)
print(f"Train: {len(train_ds)} windows ({train_ds.plasmid_count} plasmids, "
      f"{data_cfg.get('train_windows_per_plasmid', 1)} windows/plasmid)")
print(f"Val:   {len(val_ds)} windows ({val_ds.plasmid_count} plasmids)")

# Class weights
class_weights = None
if train_cfg.get('class_weighted_loss', False):
    w = compute_class_weights(train_ds.labels, num_labels)
    class_weights = torch.tensor(w, dtype=torch.float32)
    print(f"Class weights enabled (inverse frequency)")

# Oversampler for rare classes
sampler = None
if train_cfg.get('oversample_rare_classes', False):
    sampler = build_oversampler(train_ds)
    print("Rare-class oversampling enabled (WeightedRandomSampler)")


# Custom trainer with class-weighted loss + oversampling
class WeightedOversamplingTrainer(Trainer):
    def __init__(self, *args, class_weights=None, train_sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights
        self._train_sampler = train_sampler

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
        weight = self._class_weights.to(logits.device) if self._class_weights is not None else None
        loss = nn.functional.cross_entropy(logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss

    def _get_train_sampler(self):
        if self._train_sampler is not None:
            return self._train_sampler
        return super()._get_train_sampler()


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.asarray(logits).argmax(-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'macro_f1': float(f1_score(labels, preds, average='macro', zero_division=0)),
    }


# Training arguments
output_dir = train_cfg['output_dir']
args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=train_cfg['num_train_epochs'],
    per_device_train_batch_size=train_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size=train_cfg['per_device_eval_batch_size'],
    learning_rate=train_cfg['learning_rate'],
    weight_decay=train_cfg.get('weight_decay', 0.0),
    warmup_ratio=train_cfg.get('warmup_ratio', 0.0),
    lr_scheduler_type=train_cfg.get('lr_scheduler_type', 'cosine'),
    eval_strategy=train_cfg.get('eval_strategy', 'epoch'),
    save_strategy=train_cfg.get('save_strategy', 'epoch'),
    save_total_limit=train_cfg.get('save_total_limit', 3),
    load_best_model_at_end=train_cfg.get('load_best_model_at_end', True),
    metric_for_best_model=train_cfg.get('metric_for_best_model', 'macro_f1'),
    greater_is_better=train_cfg.get('greater_is_better', True),
    fp16=train_cfg.get('fp16', True),
    seed=train_cfg.get('seed', 42),
    logging_steps=50,
    report_to=[],
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
)

trainer = WeightedOversamplingTrainer(
    model=loaded.model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    train_sampler=sampler,
)

print(f"\n{'='*60}")
print(f"Starting training: {train_cfg['num_train_epochs']} epochs")
print(f"  Effective samples/epoch: {len(train_ds):,}")
print(f"  Window size: {data_cfg['window_size']} bp")
print(f"  LR schedule: {train_cfg.get('lr_scheduler_type', 'cosine')}")
print(f"  Oversampling: {train_cfg.get('oversample_rare_classes', False)}")
print(f"  Checkpoints: {output_dir}")
if RESUME_FROM:
    print(f"  Resuming from: {RESUME_FROM}")
print(f"{'='*60}\n")

trainer.train(resume_from_checkpoint=RESUME_FROM)

## 6. Save best model

In [ ]:
# ====== SAVE BEST MODEL ======
best_dir = os.path.join(DRIVE_CHECKPOINT_DIR, 'best_v2')
os.makedirs(best_dir, exist_ok=True)

trainer.save_model(best_dir)
loaded.tokenizer.save_pretrained(best_dir)

# Save label names alongside the model so predict.py can find them
with open(os.path.join(best_dir, 'label_names.json'), 'w') as f:
    json.dump(label_names, f, indent=2)

# Save final val metrics
val_metrics = trainer.evaluate()
with open(os.path.join(best_dir, 'val_metrics.json'), 'w') as f:
    json.dump(val_metrics, f, indent=2)

print(f"\n\u2705 Best model (v2) saved to: {best_dir}")
print(f"Val metrics: {json.dumps(val_metrics, indent=2)}")
print(f"\nThis checkpoint is on your Google Drive and will persist after Colab shuts down.")

## 7. Evaluate on test set

In [ ]:
# ====== TEST-SET EVALUATION ======
from plasmid_host_range.evaluate import evaluate_checkpoint

metrics = evaluate_checkpoint(
    checkpoint_dir=best_dir,
    processed_dir='data/processed',
    reports_dir=DRIVE_REPORTS_DIR,
    window_size=data_cfg['window_size'],
    max_tokens=data_cfg['max_tokens'],
    eval_windows_per_plasmid=data_cfg['eval_windows_per_plasmid'],
    batch_size=64,
    run_baseline=True,
)

print("\n" + "="*60)
print("TEST-SET RESULTS")
print("="*60)
print(f"\nDL model (NT-v2 100M):")
print(f"  Top-1 accuracy: {metrics['model']['top1_accuracy']:.3f}")
print(f"  Top-3 accuracy: {metrics['model']['top3_accuracy']:.3f}")
print(f"  Macro F1:       {metrics['model']['macro_f1']:.3f}")

if 'baseline_kmer' in metrics:
    print(f"\nBaseline (6-mer + LogReg):")
    print(f"  Top-1 accuracy: {metrics['baseline_kmer']['top1_accuracy']:.3f}")
    print(f"  Top-3 accuracy: {metrics['baseline_kmer']['top3_accuracy']:.3f}")
    print(f"  Macro F1:       {metrics['baseline_kmer']['macro_f1']:.3f}")

    delta = metrics['model']['macro_f1'] - metrics['baseline_kmer']['macro_f1']
    print(f"\nDL improvement over baseline: {delta:+.3f} macro F1")

print(f"\nFull metrics: {DRIVE_REPORTS_DIR}/test_metrics.json")
print(f"Confusion matrix: {DRIVE_REPORTS_DIR}/confusion_matrix.png")

In [ ]:
# ====== SHOW CONFUSION MATRIX ======
from IPython.display import Image, display

cm_path = os.path.join(DRIVE_REPORTS_DIR, 'confusion_matrix.png')
if os.path.exists(cm_path):
    display(Image(filename=cm_path, width=700))
else:
    print("Confusion matrix not generated (matplotlib issue). Check reports dir.")

## 8. Sample prediction

In [ ]:
# ====== SAMPLE PREDICTION ======
# Grab a random plasmid from the test set and predict its host genus.
import pandas as pd
from plasmid_host_range.predict import predict_host_genus

test_df = pd.read_parquet('data/processed/test.parquet')
sample = test_df.sample(n=1, random_state=42).iloc[0]

print(f"Test plasmid: {sample['accession']}")
print(f"True genus:   {sample['genus']}")
print(f"Sequence len: {len(sample['sequence']):,} bp")
print()

preds = predict_host_genus(
    sample['sequence'],
    model_dir=best_dir,
    top_k=5,
    window_size=data_cfg['window_size'],
    max_tokens=data_cfg['max_tokens'],
)

print("Predictions:")
for genus, score in zip(preds[0].top_genera, preds[0].scores):
    marker = " \u2705" if genus == sample['genus'] else ""
    print(f"  {genus:25s}  {score:.3f}{marker}")

## 9. Download the trained model to your Mac

The best checkpoint is already on your Google Drive at:
```
My Drive/plasmid-host-range/checkpoints/best/
```

To use it locally:

1. Open Google Drive in your browser
2. Download the `best/` folder (right-click → Download)
3. Unzip and place it at:
   ```
   ~/Desktop/plasmid-host-range/checkpoints/best/
   ```
4. Then on your Mac:
   ```bash
   cd ~/Desktop/plasmid-host-range
   conda activate plasmid
   plasmid-host-range predict path/to/plasmid.fasta --model-dir checkpoints/best --top-k 3
   ```

In [ ]:
# ====== SUMMARY ======
print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nAll outputs are on your Google Drive:")
print(f"  Best model:       {best_dir}/")
print(f"  Test metrics:     {DRIVE_REPORTS_DIR}/test_metrics.json")
print(f"  Confusion matrix: {DRIVE_REPORTS_DIR}/confusion_matrix.png")
print(f"  Processed data:   {DRIVE_DATA_DIR}/")
print(f"\nThese files persist on Google Drive after Colab shuts down.")
print(f"\nTo use the model locally, download the best/ folder from Drive")
print(f"and run: plasmid-host-range predict input.fasta --model-dir checkpoints/best")